# Libraries

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns


from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [2]:
# dispongo la carpeta con funciones e importo ternara. 
from pathlib import Path
import sys
import importlib


# Obtiene la ruta del directorio actual y sube un nivel (.parent)
raiz_proyecto = Path().resolve().parent
# Agrega la ruta al sistema si no está ya incluida
if str(raiz_proyecto) not in sys.path:
  sys.path.append(str(raiz_proyecto))

# Importa tu librería o módulo
from funciones.ternaria import ternaria



# URLs y Constantes

In [3]:
BASE_URL = os.path.join('/mnt/', 'c')
BASE_DATA_URL = os.path.join('/mnt/', 'e')
COMP_URL = os.path.join(BASE_URL, 'Users', 'marco', 'Desktop', 'MASTER', 'MASTER', 'DMEyF', 'COMP1')
DATA_FOLDER = os.path.join(BASE_DATA_URL, 'DATASETS', 'DMEyF')
DATA_CRUDO_URL = os.path.join(DATA_FOLDER, 'competencia_01_crudo.csv')
DATA_TERNARIA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')
DATA_LAGS_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria_lags.csv')
DATA_DICT_URL= os.path.join(DATA_FOLDER, 'data_dict.csv')

SEED = 230047

# ANALISIS

## LEEMOS DATOS

In [4]:
df_crudo = pd.read_csv(
    DATA_CRUDO_URL,
    dtype={'numero_de_cliente': 'int32', 'foto_mes': 'int32'}
)

In [5]:
df_crudo.head()

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,0.0,1.0,0.0,-16247.77,0.0,4056.0,15732.34,1.0,0.0,1137.81
1,12159858,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12160484,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,0.0,1.0,0.0,-31103.23,0.0,1632.0,2860.54,2.0,0.0,19858.89
3,12160591,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,0.0,1.0,0.0,-13733.44,0.0,2122.0,1419.36,3.0,0.0,1231.65
4,12160747,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,0.0,1.0,0.0,0.00,0.0,5901.0,1286.93,1.0,0.0,82.11


## Agregamos TERNARIA

In [6]:
df_ternaria = ternaria(df_crudo)

/home/marco/code/DMEyF/dmeyf2026/funciones/ternaria.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['mes_idx'] = year * 12 + month
/home/marco/code/DMEyF/dmeyf2026/funciones/ternaria.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ternaria'] = pd.Series(None, index=df.index, dtype='str')


In [7]:
df_ternaria.head()

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,ternaria
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,1.0,0.0,-16247.77,0.0,4056.0,15732.34,1.0,0.0,1137.81,continua
1,12159858,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,continua
2,12160484,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,1.0,0.0,-31103.23,0.0,1632.0,2860.54,2.0,0.0,19858.89,continua
3,12160591,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,1.0,0.0,-13733.44,0.0,2122.0,1419.36,3.0,0.0,1231.65,continua
4,12160747,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,1.0,0.0,0.00,0.0,5901.0,1286.93,1.0,0.0,82.11,continua


## LAGs

In [8]:
df_trabajo = df_ternaria.copy()

In [9]:
# Debe existir una sola fila por cliente y mes
claves = ["numero_de_cliente", "foto_mes"]

if df_trabajo.duplicated(claves).any():
    raise ValueError("Hay más de una fila para algún cliente y foto_mes")

df_trabajo = (
    df_trabajo
    .sort_values(claves)
    .reset_index(drop=True)
    .copy()
)


In [10]:

# Índice mensual: permite controlar correctamente pasos como 202112 -> 202201
foto_mes_num = pd.to_numeric(df_trabajo["foto_mes"])

anio = foto_mes_num // 100
mes = foto_mes_num % 100

if not mes.between(1, 12).all():
    raise ValueError("foto_mes contiene meses inválidos")

df_trabajo["_mes_idx"] = anio * 12 + mes

# Columnas sobre las que se generan los lags
columnas_excluidas = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria",
    "_mes_idx"
}

columnas_base = [
    columna
    for columna in df_trabajo.columns
    if columna not in columnas_excluidas
]


In [11]:

agrupado = df_trabajo.groupby("numero_de_cliente", sort=False)

# Valor de la fila anterior para el mismo cliente
lags = agrupado[columnas_base].shift(1)
mes_anterior = agrupado["_mes_idx"].shift(1)

# Sólo es válido si corresponde exactamente al mes anterior
mes_consecutivo = (df_trabajo["_mes_idx"] - mes_anterior).eq(1)
lags.loc[~mes_consecutivo, :] = np.nan

# Delta únicamente para variables numéricas
columnas_numericas = (
    df_trabajo[columnas_base]
    .select_dtypes(include="number")
    .columns
)

deltas = df_trabajo[columnas_numericas] - lags[columnas_numericas]

lags.columns = [f"{columna}_lag1" for columna in lags.columns]
deltas.columns = [f"{columna}_delta1" for columna in deltas.columns]

df_lags = pd.concat(
    [df_trabajo, lags, deltas],
    axis=1
).drop(columns="_mes_idx")

In [12]:
df_lags.head(20)

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares_delta1,Visa_fultimo_cierre_delta1,Visa_mpagado_delta1,Visa_mpagospesos_delta1,Visa_mpagosdolares_delta1,Visa_fechaalta_delta1,Visa_mconsumototal_delta1,Visa_cconsumos_delta1,Visa_cadelantosefectivo_delta1,Visa_mpagominimo_delta1
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12159854,202104,1,0,0,56,135,1622.57,27663.64,931.59,...,0.0,1.0,0.0,0.00,0.0,30.0,0.00,0.0,0.0,0.00
2,12159854,202105,1,0,0,56,136,1987.98,28738.37,934.07,...,0.0,3.0,0.0,0.00,0.0,31.0,0.00,0.0,0.0,0.00
3,12159854,202106,1,0,0,56,137,2580.75,29494.76,915.19,...,0.0,-5.0,0.0,0.00,0.0,30.0,0.00,0.0,0.0,0.00
4,12159854,202107,1,0,0,56,138,2686.85,29431.10,688.94,...,0.0,3.0,0.0,0.00,0.0,31.0,0.00,0.0,0.0,0.00
5,12159854,202108,1,0,0,56,139,2707.62,29507.13,1023.41,...,0.0,3.0,0.0,0.00,0.0,31.0,0.00,0.0,0.0,0.00
6,12159858,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,12159858,202104,1,0,0,49,103,323.58,19898.05,157.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,12159858,202105,1,0,0,49,104,3017.77,17742.80,154.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,12159858,202106,1,0,0,49,105,3290.91,16173.55,162.12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# pd.Series(df_lags.columns)[pd.Series(df_lags.columns).str.contains('mrentabilidad') ]
df_lags.loc[df_lags['numero_de_cliente']==12159858][['foto_mes','mrentabilidad','mrentabilidad_lag1', 'mrentabilidad_delta1']]
# df_lags.loc[df_lags['numero_de_cliente']==12159858][['foto_mes','mcomisiones','mcomisiones_lag1', 'mcomisiones_delta1']]
# df_lags.loc[df_lags['numero_de_cliente']==12159858]

,foto_mes,mrentabilidad,mrentabilidad_lag1,mrentabilidad_delta1
6,202103,78.43,NaN,NaN
7,202104,323.58,78.43,245.15
8,202105,3017.77,323.58,2694.19
9,202106,3290.91,3017.77,273.14
10,202107,501.84,3290.91,-2789.07
11,202108,319.34,501.84,-182.50


In [14]:
# guardamos como csv
# df_lags.to_csv(DATA_LAGS_URL, index=False)

# LAGs fracionarios

In [10]:
df_trabajo = df_ternaria.copy()

In [11]:


def generar_pesos(K, factor_perdida):
    """
    factor_perdida indica qué proporción de importancia se pierde
    por cada mes adicional hacia atrás.
    """
    if K < 1:
        raise ValueError("K debe ser mayor o igual a 1")

    if not 0 <= factor_perdida < 1:
        raise ValueError("factor_perdida debe estar entre 0 y 1")

    retencion = 1 - factor_perdida
    pesos = retencion ** np.arange(K, dtype=float)

    return pesos / pesos.sum()


def agregar_lags_ponderados(
    df,
    K=6,
    factor_perdida=0.4,
    columnas=None,
    tam_bloque=20
):
    df = df.copy()

    claves = ["numero_de_cliente", "foto_mes"]

    if df.duplicated(claves).any():
        raise ValueError("Hay más de una fila para algún cliente y mes")

    # Convierto YYYYMM a un índice mensual continuo.
    foto_mes = pd.to_numeric(df["foto_mes"], errors="raise").astype("int64")
    mes = foto_mes % 100

    if not mes.between(1, 12).all():
        raise ValueError("foto_mes contiene meses inválidos")

    df["_mes_idx"] = (foto_mes // 100) * 12 + mes

    # El cálculo necesita los registros ordenados cronológicamente.
    df = (
        df.sort_values(
            ["numero_de_cliente", "_mes_idx"],
            kind="stable"
        )
        .reset_index(drop=True)
    )

    if columnas is None:
        excluidas = {
            "numero_de_cliente",
            "foto_mes",
            "clase_ternaria",
            "_mes_idx"
        }

        columnas = [
            columna
            for columna in df.select_dtypes(include=np.number).columns
            if columna not in excluidas
            and not columna.endswith("_lag_pond")
            and not columna.endswith("_delta_lag_pond")
        ]

    no_numericas = [
        columna
        for columna in columnas
        if not pd.api.types.is_numeric_dtype(df[columna])
    ]

    if no_numericas:
        raise TypeError(
            f"Las siguientes columnas no son numéricas: {no_numericas}"
        )

    pesos = generar_pesos(K, factor_perdida)
    n_filas = len(df)

    # Peso correspondiente a cada observación histórica.
    # Se usa la distancia real entre meses, no simplemente la fila anterior.
    pesos_por_fila = np.zeros((n_filas, K), dtype=np.float64)

    grupo_mes = df.groupby(
        "numero_de_cliente",
        sort=False
    )["_mes_idx"]

    for desplazamiento in range(1, K + 1):
        mes_anterior = grupo_mes.shift(desplazamiento)
        distancia = df["_mes_idx"] - mes_anterior

        valido = distancia.between(1, K)
        posiciones = distancia.loc[valido].astype(int).to_numpy() - 1

        pesos_por_fila[
            valido.to_numpy(),
            desplazamiento - 1
        ] = pesos[posiciones]

    nuevas_columnas = []

    # Se procesa por bloques para limitar el uso de memoria.
    for inicio in range(0, len(columnas), tam_bloque):
        bloque = columnas[inicio:inicio + tam_bloque]

        numerador = np.zeros((n_filas, len(bloque)), dtype=np.float64)
        denominador = np.zeros_like(numerador)

        grupo = df[bloque].groupby(
            df["numero_de_cliente"],
            sort=False
        )

        for desplazamiento in range(1, K + 1):
            valores_pasados = grupo.shift(desplazamiento).to_numpy(
                dtype=np.float64,
                na_value=np.nan
            )

            peso = pesos_por_fila[
                :, desplazamiento - 1
            ][:, None]

            disponible = valores_pasados.notna() if False else ~np.isnan(
                valores_pasados
            )

            numerador += np.where(
                disponible,
                valores_pasados * peso,
                0.0
            )

            denominador += disponible * peso

        lag_ponderado = np.divide(
            numerador,
            denominador,
            out=np.full_like(numerador, np.nan),
            where=denominador > 0
        )

        valores_actuales = df[bloque].to_numpy(
            dtype=np.float64,
            na_value=np.nan
        )

        delta_ponderado = valores_actuales - lag_ponderado

        nuevas_columnas.append(
            pd.DataFrame(
                lag_ponderado,
                columns=[f"{c}_lag_pond" for c in bloque]
            )
        )

        nuevas_columnas.append(
            pd.DataFrame(
                delta_ponderado,
                columns=[f"{c}_delta_lag_pond" for c in bloque]
            )
        )

    df = pd.concat([df, *nuevas_columnas], axis=1)

    return df.drop(columns="_mes_idx")

In [12]:
df_trabajo.shape

(983061, 155)

In [13]:
k = 6
factor_perdida = .4
print(generar_pesos(3, 0.4))
df_lags_fraccionarios = agregar_lags_ponderados(
    df_trabajo,
    K=k,
    factor_perdida=factor_perdida
)

[0.51020408 0.30612245 0.18367347]


In [14]:
df_lags_fraccionarios

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares_delta_lag_pond,Visa_fultimo_cierre_delta_lag_pond,Visa_mpagado_delta_lag_pond,Visa_mpagospesos_delta_lag_pond,Visa_mpagosdolares_delta_lag_pond,Visa_fechaalta_delta_lag_pond,Visa_mconsumototal_delta_lag_pond,Visa_cconsumos_delta_lag_pond,Visa_cadelantosefectivo_delta_lag_pond,Visa_mpagominimo_delta_lag_pond
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12159854,202104,1,0,0,56,135,1622.57,27663.64,931.59,...,0.0,1.000000,0.0,0.000000e+00,0.0,30.000000,0.0,0.0,0.0,0.0
2,12159854,202105,1,0,0,56,136,1987.98,28738.37,934.07,...,0.0,3.375000,0.0,1.818989e-12,0.0,42.250000,0.0,0.0,0.0,0.0
3,12159854,202106,1,0,0,56,137,2580.75,29494.76,915.19,...,0.0,-3.346939,0.0,1.818989e-12,0.0,50.693878,0.0,0.0,0.0,0.0
4,12159854,202107,1,0,0,56,138,2686.85,29431.10,688.94,...,0.0,1.191176,0.0,3.637979e-12,0.0,58.397059,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983056,78249586,202108,0,0,0,37,1,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983057,78249850,202108,1,0,0,35,1,0.53,0.53,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983058,78250419,202108,1,0,0,24,1,116.39,116.39,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983059,78253625,202108,0,0,0,28,1,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df_lags_fraccionarios.columns

Index(['numero_de_cliente', 'foto_mes', 'active_quarter', 'cliente_vip',
       'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad',
       'mrentabilidad_annual', 'mcomisiones',
       ...
       'Visa_madelantodolares_delta_lag_pond',
       'Visa_fultimo_cierre_delta_lag_pond', 'Visa_mpagado_delta_lag_pond',
       'Visa_mpagospesos_delta_lag_pond', 'Visa_mpagosdolares_delta_lag_pond',
       'Visa_fechaalta_delta_lag_pond', 'Visa_mconsumototal_delta_lag_pond',
       'Visa_cconsumos_delta_lag_pond',
       'Visa_cadelantosefectivo_delta_lag_pond',
       'Visa_mpagominimo_delta_lag_pond'],
      dtype='str', length=459)

In [16]:

df_lags_fraccionarios.loc[df_lags_fraccionarios['numero_de_cliente']==12159858][['foto_mes','mrentabilidad','mrentabilidad_lag_pond', 'mrentabilidad_delta_lag_pond']]
# df_lags_fraccionarios.loc[df_lags_fraccionarios['numero_de_cliente']==12159858]

,foto_mes,mrentabilidad,mrentabilidad_lag_pond,mrentabilidad_delta_lag_pond
6,202103,78.43,NaN,NaN
7,202104,323.58,78.430000,245.150000
8,202105,3017.77,231.648750,2786.121250
9,202106,3290.91,1653.139184,1637.770816
10,202107,501.84,2405.791213,-1903.951213
11,202108,319.34,1579.996967,-1260.656967


In [17]:
filename = f'competencia_01_ternaria_lf_k{k}_perdida{factor_perdida*10:.0f}.csv'
df_lags_fraccionarios.to_csv(os.path.join(DATA_FOLDER, filename), index=False)

In [18]:
df_lags_fraccionarios

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares_delta_lag_pond,Visa_fultimo_cierre_delta_lag_pond,Visa_mpagado_delta_lag_pond,Visa_mpagospesos_delta_lag_pond,Visa_mpagosdolares_delta_lag_pond,Visa_fechaalta_delta_lag_pond,Visa_mconsumototal_delta_lag_pond,Visa_cconsumos_delta_lag_pond,Visa_cadelantosefectivo_delta_lag_pond,Visa_mpagominimo_delta_lag_pond
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12159854,202104,1,0,0,56,135,1622.57,27663.64,931.59,...,0.0,1.000000,0.0,0.000000e+00,0.0,30.000000,0.0,0.0,0.0,0.0
2,12159854,202105,1,0,0,56,136,1987.98,28738.37,934.07,...,0.0,3.375000,0.0,1.818989e-12,0.0,42.250000,0.0,0.0,0.0,0.0
3,12159854,202106,1,0,0,56,137,2580.75,29494.76,915.19,...,0.0,-3.346939,0.0,1.818989e-12,0.0,50.693878,0.0,0.0,0.0,0.0
4,12159854,202107,1,0,0,56,138,2686.85,29431.10,688.94,...,0.0,1.191176,0.0,3.637979e-12,0.0,58.397059,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983056,78249586,202108,0,0,0,37,1,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983057,78249850,202108,1,0,0,35,1,0.53,0.53,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983058,78250419,202108,1,0,0,24,1,116.39,116.39,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983059,78253625,202108,0,0,0,28,1,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
